# 10 · ephys figures: publication-style figures and provisional drug statistics

Descriptive figures of the intrinsic properties of PMvDAT+ neurons: every cell shown, n in every panel, means only
together with the individual cells. The only statistics are the **PROVISIONAL** paired drug tests (step 4), to be
reviewed before reporting (`README.md`, "Drug statistics: what and why").

| Step | What happens |
|---|---|
| 1 | load the outputs of 02 and 05-08; intrinsic labels |
| 2 | summary table (n, mean ± SD, median [IQR]) |
| 3 | figures 1-5: examples, labels, properties, AP at ramp rheobase, excitability |
| 4 | provisional drug statistics and figure 6 per drug condition |
| 5 | save |

Membrane potentials are not corrected for the liquid junction potential (+15.9 mV).

In [ ]:
OVERWRITE = True

import pandas as pd
from IPython.display import display
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 250)

from ephys_analysis import figures as fg
from ephys_analysis.baseline import save_figure
from ephys_analysis.cell_overview import cell_labels
from ephys_analysis.drug_stats import CAVEAT, DrugStatsSettings, drug_statistics, peak_sham_table, select_applications
from ephys_analysis.ramp import RampSettings
from ephys_analysis.steps import StepSettings
from ephys_analysis.timecourse import condition_slug
from ephys_analysis.utils import RunLog, check_writable, find_repo_root, load_config, repo_path, stamp, write_csv

REPO = find_repo_root()
cfg = load_config(REPO / "10_ephys_figures")
fg.configure(cfg)
out = REPO / "10_ephys_figures" / "output"
figs = out / "figures"
fmt = tuple(cfg["figure_formats"])
targets = [out / "summary_table.csv", out / "drug_stats.csv", out / "drug_responders.csv"]
check_writable(targets, OVERWRITE)
log = RunLog(cfg, out)

## Step 1 · Inputs and labels
One recording per cell and protocol, as selected by 08 (`selection.csv`).

In [ ]:
data = fg.load_inputs(cfg, repo_path)
summary, selection = data["summary"], data["selection"]
labels = cell_labels(summary, cfg["labels"])
display(labels)

## Step 2 · Summary table

In [ ]:
table = fg.summary_table(summary, cfg["properties"] + cfg["ap_properties"])
display(table.drop(columns="note").round(3))

## Step 3 · Figures 1-5
- **fig1** example traces; **fig2** silent / tonic, block, rebound (count bars and a per-cell dot matrix);
- **fig3** properties, one dot per cell (median, IQR); **fig4** AP at ramp rheobase (average AP, phase plane);
- **fig5** F–I per cell (open = block), block onset (cells without block above the largest step), rebound vs sag.

In [ ]:
rows = fg.example_rows(cfg.get("examples"), labels, selection, data["ready"])
fig = fg.fig_examples(rows, labels, StepSettings(), RampSettings(), None, fmt, cfg.get("example_spontaneous_s", 30))
display(fig); save_figure(fig, figs / "fig1_examples", fmt)
fig = fg.fig_labels(labels)
display(fig); save_figure(fig, figs / "fig2_labels", fmt)
fig = fg.fig_properties(summary, labels, cfg["properties"])
display(fig); save_figure(fig, figs / "fig3_properties", fmt)
fig = fg.fig_ap(data["waves"], summary, labels, fg.recordings_of(selection, "ramp"), cfg["ap_properties"])
display(fig); save_figure(fig, figs / "fig4_ap", fmt)
fig = fg.fig_excitability(data["step_sweeps"], summary, labels, fg.recordings_of(selection, "steps"))
display(fig); save_figure(fig, figs / "fig5_excitability", fmt)

## Step 4 · Drug statistics (PROVISIONAL) and figures 6-7
**Estimation statistics first** (Ho et al. 2019, DABEST style): the paired effect size (mean difference, median
difference, Hedges' g) with a **BCa bootstrap 95 % CI** is the headline; the estimation plot shows every cell and
the bootstrap distribution of the difference. Secondary, in `drug_stats.csv`: Wilcoxon / paired t, Friedman with
Holm-corrected pairwise tests, Holm-corrected secondary outcomes, exact McNemar on silent ↔ tonic, responders with a
Clopper–Pearson CI, the peak only against a sham peak. Below 5 cells: no CI or p-value. **Review every rule before
reporting** (README).

In [ ]:
s = DrugStatsSettings.from_config(cfg["drug_stats"])
stats_df, resp = drug_statistics(data["tc_summary"], data["tc_bins"], s, selection, summary)
print(CAVEAT)
display(stats_df[["condition", "role", "outcome", "comparison", "n", "mean_diff", "ci_low", "ci_high", "p_wilcoxon",
                  "p_holm", "note"]].round(4))
apps = select_applications(data["tc_summary"], selection)
for cond, g in apps.groupby("condition"):
    fig = fg.fig_drug(cond, g, data["tc_bins"], labels, stats_df[stats_df["condition"] == cond],
                      tuple(cfg["drug_window_s"]), None, fmt, peak_sham_table(g, data["tc_bins"], s), s)
    display(fig); save_figure(fig, figs / f"fig6_drug_{condition_slug(cond)}", fmt)
    if s.estimation_secondary:
        fig = fg.fig_estimation_secondary(cond, g, labels, list(s.secondary), s)
        display(fig); save_figure(fig, figs / f"fig7_estimation_{condition_slug(cond)}", fmt)

## Step 5 · Save

In [ ]:
for df, p in zip((table, stats_df, resp), targets):
    write_csv(stamp(df, log.run_id), p)
    print("written:", p)
log.finish(targets)